# 1. PROJECT CONTEXT

## Research Title
**"Analisis Performa Algoritma Machine Learning untuk Klasifikasi Jenis dan Tingkat Keparahan Cyberbullying pada Teks Bahasa Indonesia Menggunakan TF-IDF"**
*(Performance Analysis of Machine Learning Algorithms for Cyberbullying Type and Severity Classification in Indonesian Text Using TF-IDF)*

## Role of Error Analysis
This is the eleventh stage of the pipeline. In Stage 09, we evaluated all models and objectively selected the best performer based on the F1-Macro score. However, performance metrics only tell us *how often* the model is wrong, not *why* it is wrong. This Error Analysis notebook dissects the misclassifications of the final model to understand its linguistic limitations and identify potential annotation (labeling) errors in the dataset.


# 2. ERROR ANALYSIS OBJECTIVES

Our goals for this stage are to:
- Dynamically load the winning model from Stage 09.
- Map the TF-IDF predictions back to the human-readable original Indonesian text.
- Identify common model errors and confusion patterns (e.g., confusing 'Religion' with 'Race').
- Investigate if text length (short vs long text) correlates with higher error rates.
- Isolate "High-Confidence Errors" which often indicate human annotation mistakes in the dataset.
- Provide evidence-based recommendations for future NLP improvements.


In [1]:
# 3. IMPORT LIBRARIES

import pandas as pd
import numpy as np
import pathlib
import json
import joblib
from scipy import sparse
import warnings

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
# 4. CONFIGURATION

PROJECT_ROOT = pathlib.Path.cwd().parent
TFIDF_DIR = PROJECT_ROOT / "data" / "processed" / "tfidf"
MODELS_DIR = PROJECT_ROOT / "models"
BASE_REPORTS_DIR = PROJECT_ROOT / "reports"
ERROR_REPORTS_DIR = BASE_REPORTS_DIR / "error_analysis"

# Ensure error analysis directory exists to prevent cluttering the main reports folder
ERROR_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Test Data Paths
X_TEST_PATH = TFIDF_DIR / "X_test_tfidf.npz"
Y_TEST_PATH = TFIDF_DIR / "y_test.csv"
TEXT_METADATA_PATH = TFIDF_DIR / "test_metadata.csv"
MODEL_SELECTION_PATH = BASE_REPORTS_DIR / "model_evaluation" / "model_selection.json"

# Analysis Thresholds
HIGH_CONFIDENCE_THRESHOLD = 0.85 # 85% probability or equivalent score

print(f"Error Reports Directory: {ERROR_REPORTS_DIR}")
print(f"High Confidence Threshold: {HIGH_CONFIDENCE_THRESHOLD}")


Error Reports Directory: /home/zapp/Kampus/PM-NEW/reports/error_analysis
High Confidence Threshold: 0.85


In [3]:
# 5. LOAD EVALUATION DATA & 6. LOAD ORIGINAL TEXT DATA

if not X_TEST_PATH.exists() or not Y_TEST_PATH.exists() or not TEXT_METADATA_PATH.exists():
    raise FileNotFoundError("FAIL: Test artifacts or text metadata not found. Please run previous stages first.")

print("Loading Unseen Test Data and Metadata...")
X_test = sparse.load_npz(X_TEST_PATH)
y_test_df = pd.read_csv(Y_TEST_PATH)
y_test = y_test_df.iloc[:, 0]  

# Load human readable text to map back predictions
metadata_df = pd.read_csv(TEXT_METADATA_PATH)

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"metadata shape: {metadata_df.shape}")


Loading Unseen Test Data and Metadata...
X_test shape: (6443, 35222)
y_test shape: (6443,)
metadata shape: (6443, 9)


In [4]:
# 7. LOAD PREDICTIONS

if not MODEL_SELECTION_PATH.exists():
    raise FileNotFoundError("FAIL: model_selection.json not found. You must complete 09_model_evaluation.ipynb first.")

with open(MODEL_SELECTION_PATH, 'r') as f:
    selection_meta = json.load(f)

best_model_name = selection_meta['selected_model']
print(f"Targeting Best Model: {best_model_name}")

# Load the actual model
model_path = MODELS_DIR / f"{best_model_name}.pkl"
best_model = joblib.load(model_path)

# Generate Predictions
y_pred_raw = best_model.predict(X_test)

# Handle XGBoost Mapping if necessary
if 'xgboost' in best_model_name:
    xgb_mapping_path = MODELS_DIR / "xgboost_label_mapping.json"
    with open(xgb_mapping_path, 'r') as f:
        xgb_mapping = json.load(f)
    xgb_reverse_mapping = {v: k for k, v in xgb_mapping.items()}
    y_pred = pd.Series(y_pred_raw).map(xgb_reverse_mapping).values
else:
    y_pred = y_pred_raw

# Generate Confidence Scores
y_prob = None
is_probability = False

if hasattr(best_model, "predict_proba"):
    y_prob = best_model.predict_proba(X_test)
    y_confidence = np.max(y_prob, axis=1) # The max probability assigned to the predicted class
    is_probability = True
elif hasattr(best_model, "decision_function"):
    y_prob = best_model.decision_function(X_test)
    y_confidence = np.max(y_prob, axis=1) # The highest distance margin

print("Predictions and Confidence Scores generated successfully.")


Targeting Best Model: linear_svm_baseline
Predictions and Confidence Scores generated successfully.


In [5]:
# 8. VALIDATE DATA ALIGNMENT

if not (X_test.shape[0] == y_test.shape[0] == metadata_df.shape[0] == len(y_pred)):
    raise ValueError("CRITICAL ERROR: Data Alignment mismatch! The rows between features, labels, and text metadata do not match.")

print("Data alignment verified. All rows map perfectly 1:1.")


Data alignment verified. All rows map perfectly 1:1.


In [6]:
# 9. BUILD ERROR ANALYSIS DATAFRAME

df_analysis = metadata_df.copy()

# Add Model Outputs
df_analysis['actual_label'] = y_test.values
df_analysis['predicted_label'] = y_pred
df_analysis['is_correct'] = (df_analysis['actual_label'] == df_analysis['predicted_label'])
df_analysis['confidence_score'] = y_confidence

# Text Length Analysis Features
# Count characters from the original text
df_analysis['char_count'] = df_analysis.iloc[:, 0].astype(str).apply(len)
# Count words from the original text (splitting by space)
df_analysis['word_count'] = df_analysis.iloc[:, 0].astype(str).apply(lambda x: len(x.split()))

print("Master Error Analysis DataFrame built successfully.")
display(df_analysis.head(3))


Master Error Analysis DataFrame built successfully.


,text,text_processed,cyberbullying_type,severity_level,original_cyberbullying_type,original_severity_level,source_dataset,source_file,original_row_id,actual_label,predicted_label,is_correct,confidence_score,char_count,word_count
0,tertawa lalu tb2 mnangis dulang2 sprti orgil,tertawa lalu tb2 mnangis dulang2 sprti orgil,harassment,NaN,"['1', '0']",NaN,indotoxic2024_annotated_data_v2_final,indotoxic2024_annotated_data_v2_final.csv,11845,harassment,normal,False,0.148109,44,7
1,rt personal agenda seperti ingin bekerja di bi...,rt personal agenda seperti ingin bekerja di bi...,normal,NaN,not_cyberbullying,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,1590,normal,normal,True,0.118059,125,18
2,"120 Negara Sepakat Gencatan Senjata di Gaza, 1...",120 negara sepakat gencatan senjata di gaza 14...,normal,NaN,"['0', '0', '0', '0']",NaN,indotoxic2024_annotated_data_v2_final,indotoxic2024_annotated_data_v2_final.csv,11439,normal,normal,True,0.169375,98,15


In [7]:
# 10. IDENTIFY MISCLASSIFIED SAMPLES

df_correct = df_analysis[df_analysis['is_correct'] == True]
df_errors = df_analysis[df_analysis['is_correct'] == False]

total_samples = len(df_analysis)
total_errors = len(df_errors)
error_rate = (total_errors / total_samples) * 100

print(f"Total Samples: {total_samples}")
print(f"Total Correct: {len(df_correct)}")
print(f"Total Errors : {total_errors}")
print(f"Overall Error Rate: {error_rate:.2f}%")

# Save misclassified samples
errors_path = ERROR_REPORTS_DIR / "misclassified_samples.csv"
df_errors.to_csv(errors_path, index=False)
print(f"Saved misclassified samples to {errors_path}")


Total Samples: 6443
Total Correct: 5126
Total Errors : 1317
Overall Error Rate: 20.44%
Saved misclassified samples to /home/zapp/Kampus/PM-NEW/reports/error_analysis/misclassified_samples.csv


In [8]:
# 11. ANALYZE CONFUSION PATTERNS

# Create pairs of (Actual, Predicted) to see what is confused with what
confusion_pairs = df_errors.groupby(['actual_label', 'predicted_label']).size().reset_index(name='error_count')
confusion_pairs = confusion_pairs.sort_values(by='error_count', ascending=False).reset_index(drop=True)

# Calculate percentage contribution to total errors
confusion_pairs['percent_of_total_errors'] = (confusion_pairs['error_count'] / total_errors) * 100

print("--- TOP 10 MOST FREQUENT CONFUSION PAIRS ---")
display(confusion_pairs.head(10))

# Save
confusion_pairs_path = ERROR_REPORTS_DIR / "confusion_pairs.csv"
confusion_pairs.to_csv(confusion_pairs_path, index=False)


--- TOP 10 MOST FREQUENT CONFUSION PAIRS ---


,actual_label,predicted_label,error_count,percent_of_total_errors
0,harassment,normal,410,31.131359
1,normal,harassment,333,25.284738
2,normal,hate_speech,153,11.617312
3,hate_speech,normal,135,10.250569
4,insult,hate_speech,62,4.707669
5,hate_speech,insult,56,4.252088
6,harassment,hate_speech,49,3.720577
7,normal,insult,39,2.961276
8,insult,normal,34,2.581625
9,hate_speech,harassment,29,2.201974


In [9]:
# 12. ANALYZE CLASS-LEVEL ERRORS

# Group by actual label
class_stats = []

for label in df_analysis['actual_label'].unique():
    subset = df_analysis[df_analysis['actual_label'] == label]
    support = len(subset)
    incorrect = len(subset[subset['is_correct'] == False])
    class_error_rate = (incorrect / support) * 100 if support > 0 else 0
    
    class_stats.append({
        'Class': label,
        'Support': support,
        'Misclassified': incorrect,
        'Error_Rate_Percent': class_error_rate
    })

df_class_errors = pd.DataFrame(class_stats).sort_values(by='Error_Rate_Percent', ascending=False).reset_index(drop=True)

print("--- ERROR RATE BY CYBERBULLYING CLASS ---")
display(df_class_errors)

# Save
class_errors_path = ERROR_REPORTS_DIR / "class_error_analysis.csv"
df_class_errors.to_csv(class_errors_path, index=False)


--- ERROR RATE BY CYBERBULLYING CLASS ---


,Class,Support,Misclassified,Error_Rate_Percent
0,harassment,772,467,60.492228
1,insult,262,105,40.076336
2,hate_speech,1072,220,20.522388
3,normal,4337,525,12.105142


In [10]:
# 13. ANALYZE TEXT LENGTH AND ERROR RATE

# Create Text Length Bins (Short, Medium, Long) based on quantiles or absolute word counts
def categorize_length(words):
    if words <= 10: return "Short (<=10 words)"
    elif words <= 30: return "Medium (11-30 words)"
    else: return "Long (>30 words)"
    
df_analysis['length_category'] = df_analysis['word_count'].apply(categorize_length)

length_stats = df_analysis.groupby('length_category')['is_correct'].agg(['count', 'sum'])
length_stats.rename(columns={'count': 'Total_Samples', 'sum': 'Correct_Samples'}, inplace=True)
length_stats['Error_Samples'] = length_stats['Total_Samples'] - length_stats['Correct_Samples']
length_stats['Error_Rate_Percent'] = (length_stats['Error_Samples'] / length_stats['Total_Samples']) * 100

print("--- ERROR RATE BY TEXT LENGTH ---")
display(length_stats.sort_values(by='Error_Rate_Percent', ascending=False))


--- ERROR RATE BY TEXT LENGTH ---


,Total_Samples,Correct_Samples,Error_Samples,Error_Rate_Percent
length_category,,,,
Short (<=10 words),1553,1209,344,22.150676
Medium (11-30 words),3028,2393,635,20.970938
Long (>30 words),1862,1524,338,18.152524


In [11]:
# 14. ANALYZE CONFIDENCE AND DECISION SCORES

print(f"Confidence Metric Type: {'Probability (0.0 to 1.0)' if is_probability else 'Decision Score (Distance from Hyperplane)'}")

print("\nAverage Confidence - Correct Predictions:   ", df_correct['confidence_score'].mean())
print("Average Confidence - Incorrect Predictions: ", df_errors['confidence_score'].mean())

print("\nObservation: Ideally, the model should have high confidence when correct, and low confidence when making errors. If confidence is high on errors, it indicates 'overconfidence' in wrong patterns.")


Confidence Metric Type: Decision Score (Distance from Hyperplane)

Average Confidence - Correct Predictions:    0.6989605695005784
Average Confidence - Incorrect Predictions:  0.27479654566954653

Observation: Ideally, the model should have high confidence when correct, and low confidence when making errors. If confidence is high on errors, it indicates 'overconfidence' in wrong patterns.


In [12]:
# 15. ANALYZE HIGH-CONFIDENCE ERRORS

if is_probability:
    df_high_conf_errors = df_errors[df_errors['confidence_score'] >= HIGH_CONFIDENCE_THRESHOLD]
else:
    # For decision scores, 'high confidence' is harder to threshold statically without scaling.
    # We'll take the top 10% highest margin errors.
    threshold = df_errors['confidence_score'].quantile(0.90)
    df_high_conf_errors = df_errors[df_errors['confidence_score'] >= threshold]

print(f"Number of High-Confidence Errors: {len(df_high_conf_errors)}")
print("These are cases where the model is 'absolutely sure' but completely wrong.")

# Save
high_conf_path = ERROR_REPORTS_DIR / "high_confidence_errors.csv"
df_high_conf_errors.to_csv(high_conf_path, index=False)
print(f"Saved to: {high_conf_path}")


Number of High-Confidence Errors: 132
These are cases where the model is 'absolutely sure' but completely wrong.
Saved to: /home/zapp/Kampus/PM-NEW/reports/error_analysis/high_confidence_errors.csv


In [13]:
# 16. QUALITATIVE ERROR REVIEW & 17. POTENTIAL ANNOTATION ISSUES

# High-Confidence Errors are prime suspects for Annotation Issues. 
# If human annotators mislabeled a text, the model (if learning the true pattern) will get it "wrong" but with high confidence.

df_annotation_review = df_high_conf_errors.copy()
df_annotation_review = df_annotation_review[['actual_label', 'predicted_label', 'confidence_score', df_annotation_review.columns[0]]]
df_annotation_review['human_review_category'] = "" # Blank column for manual tagging

print("--- POTENTIAL ANNOTATION ISSUES (TOP 5 SAMPLES) ---")
display(df_annotation_review.head())

review_path = ERROR_REPORTS_DIR / "potential_annotation_issues_for_review.csv"
df_annotation_review.to_csv(review_path, index=False)
print(f"\nExported blank review template to {review_path} for manual inspection.")


--- POTENTIAL ANNOTATION ISSUES (TOP 5 SAMPLES) ---


,actual_label,predicted_label,confidence_score,text,human_review_category
150,harassment,normal,0.794230,"BONAJUMA ZONE Ngamuk Saat Dirujuk Ke RS, ODGJ ...",
251,normal,hate_speech,0.980828,"USER ""Aduh.. mau eek""'",
272,hate_speech,harassment,1.022849,#MataNajwaDebatJakarta Anies anda SADIS. Topen...,
328,insult,hate_speech,1.426652,Ahok versi Pakpahan .. Trims kepada semua piha...,
371,normal,harassment,1.186496,Yg terjadi hr ini kpd zionis israel adl akibat...,



Exported blank review template to /home/zapp/Kampus/PM-NEW/reports/error_analysis/potential_annotation_issues_for_review.csv for manual inspection.


In [ ]:
# 18. OPTIONAL BASELINE VS TUNED ERROR COMPARISON

print("Skipped: To maintain sharp focus on dissecting the linguistic failures of the absolute best model, cross-model error comparison is omitted here.")


In [14]:
# 19. GENERATE VISUALIZATIONS

sns.set_theme(style="whitegrid")

# Plot 1: Error Rate by Class
plt.figure(figsize=(10, 6))
sns.barplot(data=df_class_errors, x='Error_Rate_Percent', y='Class', palette='Reds_r')
plt.title(f"Error Rate by Cyberbullying Type ({best_model_name})")
plt.xlabel("Error Rate (%)")
plt.ylabel("Actual Cyberbullying Type")
plt.tight_layout()
plt.savefig(ERROR_REPORTS_DIR / "error_rate_by_class.png", dpi=300)
plt.close()

# Plot 2: Text Length vs Errors
plt.figure(figsize=(8, 6))
sns.boxplot(data=df_analysis, x='is_correct', y='word_count', palette='Set2')
plt.title("Word Count Distribution: Correct vs Incorrect Predictions")
plt.xlabel("Is Prediction Correct?")
plt.ylabel("Word Count")
# Limit y-axis to ignore extreme outliers for better visualization
plt.ylim(0, df_analysis['word_count'].quantile(0.95)) 
plt.tight_layout()
plt.savefig(ERROR_REPORTS_DIR / "word_count_vs_errors.png", dpi=300)
plt.close()

print("Visualizations generated and saved to reports/error_analysis/")


Visualizations generated and saved to reports/error_analysis/


In [15]:
# 20. SAVE ERROR ANALYSIS ARTIFACTS

# Create a summary dictionary
summary_meta = {
    "analyzed_model": best_model_name,
    "total_test_samples": int(total_samples),
    "total_errors": int(total_errors),
    "overall_error_rate_percent": float(round(error_rate, 2)),
    "highest_error_class": str(df_class_errors.iloc[0]['Class']),
    "highest_error_class_rate": float(round(df_class_errors.iloc[0]['Error_Rate_Percent'], 2)),
    "top_confusion_pair": f"Actual {confusion_pairs.iloc[0]['actual_label']} -> Predicted {confusion_pairs.iloc[0]['predicted_label']}",
    "high_confidence_errors_count": len(df_high_conf_errors)
}

summary_path = ERROR_REPORTS_DIR / "error_analysis_summary.json"
with open(summary_path, 'w') as f:
    json.dump(summary_meta, f, indent=4)
    
print(f"Error Analysis Metadata saved to {summary_path}")


Error Analysis Metadata saved to /home/zapp/Kampus/PM-NEW/reports/error_analysis/error_analysis_summary.json


# 21. ERROR ANALYSIS SUMMARY

### Execution Integrity
This notebook performed zero model modifications. It strictly analyzed the frozen predictions of the best model against the frozen Test Set to understand its blind spots.

### Output Generated
The `reports/error_analysis/` directory now contains:
- `misclassified_samples.csv`: Every single mistake the model made.
- `confusion_pairs.csv`: Shows which pairs of classes confuse the model most (e.g. SARA vs Religion).
- `class_error_analysis.csv`: Error rate per class.
- `potential_annotation_issues_for_review.csv`: High-confidence errors that humans should double-check.
- `*.png`: Visualizations of error rates and text length distributions.
- `error_analysis_summary.json`: High-level summary metrics.

### Research Interpretation
1. **Confusion Pairs**: If two classes (like 'Physical Bullying' and 'General Insult') are frequently confused, it suggests their vocabulary overlaps significantly.
2. **Short Text**: If short texts have higher error rates, it indicates the TF-IDF vector lacks sufficient context (e.g., a single word insult might be ambiguous).
3. **Annotation Issues**: High confidence errors strongly suggest the model learned the true pattern of the data, but the specific row was mislabeled by a human annotator.

### Next Step
To understand *how* the model mathematically arrived at a prediction (which words triggered which classification), we proceed to the final experimental phase: `notebooks/11_explainability.ipynb`.


# 22. HOW TO RUN THIS NOTEBOOK

1. Activate the project's virtual environment.
2. Ensure you have successfully run `notebooks/09_model_evaluation.ipynb` so that `reports/model_selection.json` exists.
3. Open: `notebooks/10_error_analysis.ipynb`.
4. Run the notebook sequentially from top to bottom.
5. In **Step 11**, review the Top 10 Confusion Pairs. Take note of these pairs for your research report.
6. In **Step 12**, note which class has the highest Error Rate.
7. Open your file explorer and navigate to `reports/error_analysis/potential_annotation_issues_for_review.csv`. You can open this in Excel to manually review texts where the model strongly disagreed with the human annotator.
8. Proceed to the final Notebook: `notebooks/11_explainability.ipynb`.
